# Component 1 — M-ABSA Data Preparation Pipeline

In [12]:

from pathlib import Path
import ast
import json
import hashlib
import pandas as pd
import yaml
from IPython.display import display

CURRENT = Path.cwd().resolve()

if CURRENT.name == "notebooks" and (CURRENT.parent / "data").exists():
    PROJECT_ROOT = CURRENT.parent
elif (CURRENT / "data").exists() and (CURRENT / "notebooks").exists():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "data").exists() and (CURRENT.parent / "notebooks").exists():
    PROJECT_ROOT = CURRENT.parent
else:
    raise RuntimeError(f"Could not locate component_1 from: {CURRENT}")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "mabsa"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TAXONOMY_DIR = PROJECT_ROOT / "taxonomy"
RESULTS_DIR = PROJECT_ROOT / "results" / "eda"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_raw_files = [
    RAW_DATA_DIR / domain / "en" / f"{split}.txt"
    for domain in ["phone", "laptop"]
    for split in ["train", "dev", "test"]
]

missing = [p for p in required_raw_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing raw files:\n" + "\n".join(map(str, missing)))

print("Project root:", PROJECT_ROOT)
print("All six M-ABSA files are available.")


Project root: D:\projects\J26-DS-349\component_1
All six M-ABSA files are available.


In [13]:

def parse_mabsa_line(line):
    line = line.strip()
    if not line:
        return None

    if "####" not in line:
        return {"text": line, "triplets": [], "parse_status": "no_separator"}

    text, label_text = line.split("####", 1)

    try:
        triplets = ast.literal_eval(label_text)
        status = "ok"
    except Exception:
        triplets = []
        status = "label_parse_error"

    return {"text": text.strip(), "triplets": triplets, "parse_status": status}


def load_split(domain, split):
    path = RAW_DATA_DIR / domain / "en" / f"{split}.txt"
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            parsed = parse_mabsa_line(line)
            if parsed is None:
                continue

            rows.append({
                "review_id": f"{domain}_{split}_{line_number:05d}",
                "domain": domain,
                "split": split,
                "line_number": line_number,
                "text": parsed["text"],
                "triplets": parsed["triplets"],
                "parse_status": parsed["parse_status"],
            })

    return pd.DataFrame(rows)


reviews_df = pd.concat(
    [load_split(d, s) for d in ["phone", "laptop"] for s in ["train", "dev", "test"]],
    ignore_index=True,
)

reviews_df["num_raw_triplets"] = reviews_df["triplets"].apply(len)
reviews_df["text_sha1"] = reviews_df["text"].apply(
    lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()
)

dataset_summary = (
    reviews_df.groupby(["domain", "split"])
    .size()
    .reset_index(name="num_reviews")
)

display(dataset_summary)


,domain,split,num_reviews
0,laptop,dev,326
1,laptop,test,532
2,laptop,train,1264
3,phone,dev,307
4,phone,test,522
5,phone,train,1276


In [14]:

triplet_rows = []
invalid_triplet_count = 0

for _, row in reviews_df.iterrows():
    for triplet_index, triplet in enumerate(row["triplets"]):
        if not isinstance(triplet, (list, tuple)) or len(triplet) != 3:
            invalid_triplet_count += 1
            continue

        aspect_term, aspect_category, sentiment = triplet

        triplet_rows.append({
            "review_id": row["review_id"],
            "domain": row["domain"],
            "split": row["split"],
            "text": row["text"],
            "text_sha1": row["text_sha1"],
            "triplet_index": triplet_index,
            "aspect_term": str(aspect_term).strip(),
            "source_aspect_category": str(aspect_category).strip(),
            "sentiment": str(sentiment).strip().lower(),
        })

raw_triplets_df = pd.DataFrame(triplet_rows)

raw_triplets_df["is_null_aspect_term"] = (
    raw_triplets_df["aspect_term"].str.upper().eq("NULL")
)

sentiment_labels = sorted(raw_triplets_df["sentiment"].dropna().unique())

null_summary = (
    raw_triplets_df.groupby("domain")
    .agg(
        total_annotations=("aspect_term", "count"),
        null_aspect_terms=("is_null_aspect_term", "sum"),
        null_rate=("is_null_aspect_term", "mean"),
    )
    .reset_index()
)
null_summary["null_rate"] = (null_summary["null_rate"] * 100).round(2)

multi_summary = (
    reviews_df.assign(is_multi=reviews_df["num_raw_triplets"].gt(1))
    .groupby("domain")
    .agg(
        total_reviews=("review_id", "count"),
        multi_aspect_reviews=("is_multi", "sum"),
        multi_aspect_rate=("is_multi", "mean"),
    )
    .reset_index()
)
multi_summary["multi_aspect_rate"] = (multi_summary["multi_aspect_rate"] * 100).round(2)

display(null_summary)
display(multi_summary)
print("Sentiments:", sentiment_labels)


,domain,total_annotations,null_aspect_terms,null_rate
0,laptop,2955,627,21.22
1,phone,4810,0,0.00


,domain,total_reviews,multi_aspect_reviews,multi_aspect_rate
0,laptop,2122,592,27.90
1,phone,2105,1340,63.66


Sentiments: ['negative', 'neutral', 'positive']


In [15]:

def taxonomy_mapping_v02(domain, label):
    u = label.upper()

    phone_exact = {
        "BATTERY/LONGEVITY#CHARGING SPEED": ("charging_power", "COMMON"),
        "BATTERY/LONGEVITY#CHARGING METHOD": ("charging_power", "COMMON"),
        "BATTERY/LONGEVITY#BATTERY CAPACITY": ("battery", "COMMON"),
        "BATTERY/LONGEVITY#BATTERY LIFE": ("battery", "COMMON"),
        "BATTERY/LONGEVITY#GENERAL": ("battery", "COMMON"),
        "BATTERY/LONGEVITY#POWER CONSUMPTION SPEED": ("battery", "COMMON"),
        "BATTERY/LONGEVITY#STANDBY TIME": ("battery", "COMMON"),
        "PRODUCT CONFIGURATION#CPU": ("cpu", "COMMON"),
        "PRODUCT CONFIGURATION#MEMORY": ("memory", "COMMON"),
        "PRODUCT CONFIGURATION#OPERATING MEMORY": ("memory", "COMMON"),
        "EASE OF USE#EASY TO USE": ("usability", "COMMON"),
        "SYSTEM#SYSTEM GENERAL": ("software_system", "COMMON"),
        "SYSTEM#APPLICATION": ("software_system", "COMMON"),
        "SYSTEM#SOFTWARE COMPATIBILITY": ("software_system", "COMMON"),
        "SYSTEM#SYSTEM UPGRADE": ("software_system", "COMMON"),
        "SYSTEM#UI INTERFACE AESTHETICS": ("software_system", "COMMON"),
        "SYSTEM#LOCK SCREEN DESIGN": ("software_system", "COMMON"),
        "SYSTEM#OPERATION SMOOTHNESS": ("performance", "COMMON"),
        "SMART CONNECT#BLUETOOTH CONNECTION": ("connectivity", "COMMON"),
        "SIGNAL#WIFI SIGNAL": ("connectivity", "COMMON"),
        "PRODUCT QUALITY#FALL PROTECTION": ("durability", "COMMON"),
        "PRODUCT QUALITY#DUSTPROOF": ("durability", "COMMON"),
        "PRODUCT QUALITY#WATER RESISTANT": ("durability", "COMMON"),
        "SIGNAL#CALL QUALITY": ("call_quality", "PHONE_SPECIFIC"),
        "SIGNAL#SIGNAL GENERAL": ("mobile_network", "PHONE_SPECIFIC"),
        "SIGNAL#SIGNAL OF MOBILE NETWORK": ("mobile_network", "PHONE_SPECIFIC"),
        "SECURITY#SCREEN UNLOCK": ("biometrics_security", "PHONE_SPECIFIC"),
        "SMART CONNECT#POSITIONING AND GPS": ("gps", "PHONE_SPECIFIC"),
        "SYSTEM#NFC": ("nfc", "PHONE_SPECIFIC"),
    }

    laptop_exact = {
        "LAPTOP#PRICE": ("price", "COMMON"),
        "LAPTOP#OPERATION_PERFORMANCE": ("performance", "COMMON"),
        "LAPTOP#DESIGN_FEATURES": ("design", "COMMON"),
        "LAPTOP#USABILITY": ("usability", "COMMON"),
        "LAPTOP#CONNECTIVITY": ("connectivity", "COMMON"),
        "LAPTOP#QUALITY": ("durability", "COMMON"),
        "LAPTOP#PORTABILITY": ("portability", "LAPTOP_SPECIFIC"),
    }

    if domain == "phone" and u in phone_exact:
        a, g = phone_exact[u]
        return a, g, "yes", "confirmed"

    if domain == "laptop" and u in laptop_exact:
        a, g = laptop_exact[u]
        return a, g, "yes", "confirmed"

    phone_prefixes = [
        ("SCREEN#", "display", "COMMON"),
        ("PRICE#", "price", "COMMON"),
        ("PERFORMANCE#", "performance", "COMMON"),
        ("APPEARANCE DESIGN#", "design", "COMMON"),
        ("CAMERA#", "camera", "PHONE_SPECIFIC"),
        ("SHOOTING FUNCTIONS#", "camera", "PHONE_SPECIFIC"),
    ]

    laptop_prefixes = [
        ("POWER_SUPPLY#", "charging_power", "COMMON"),
        ("BATTERY#", "battery", "COMMON"),
        ("DISPLAY#", "display", "COMMON"),
        ("CPU#", "cpu", "COMMON"),
        ("MEMORY#", "memory", "COMMON"),
        ("OS#", "software_system", "COMMON"),
        ("SOFTWARE#", "software_system", "COMMON"),
        ("KEYBOARD#", "keyboard", "LAPTOP_SPECIFIC"),
        ("PORTS#", "ports", "LAPTOP_SPECIFIC"),
        ("FANS&COOLING#", "fans_cooling", "LAPTOP_SPECIFIC"),
        ("HARD_DISC#", "storage", "LAPTOP_SPECIFIC"),
        ("MOUSE#", "mouse", "LAPTOP_SPECIFIC"),
        ("MOTHERBOARD#", "motherboard", "LAPTOP_SPECIFIC"),
        ("OPTICAL_DRIVES#", "optical_drive", "LAPTOP_SPECIFIC"),
        ("GRAPHICS#", "graphics", "LAPTOP_SPECIFIC"),
    ]

    for prefix, aspect, group in (phone_prefixes if domain == "phone" else laptop_prefixes):
        if u.startswith(prefix):
            return aspect, group, "yes", "confirmed"

    phone_exclusions = (
        u == "OVERALL#OVERALL"
        or u.startswith("BUYER ATTITUDE#")
        or u.startswith("LOGISTICS#")
        or u.startswith("SELLER SERVICE#")
        or u.startswith("BRANDING/MARKETING#")
        or u.startswith("AFTER-SALES SERVICE#")
        or u.startswith("PRODUCT PACKAGING#")
        or u.startswith("PRODUCT ACCESSORIES#")
        or u == "EASE OF USE#AUDIENCE GROUPS"
        or u == "PRODUCT QUALITY#GENERAL"
        or u == "PRODUCT QUALITY#GENUINE PRODUCT"
        or u == "PRODUCT QUALITY#CLEANLINESS"
        or u.startswith("AUDIO/SOUND#")
        or u.startswith("KEY DESIGN#")
        or u.startswith("INTELLIGENT ASSISTANT#")
    )

    laptop_exclusions = (
        u == "LAPTOP#GENERAL"
        or u == "LAPTOP#MISCELLANEOUS"
        or u.startswith("OUT_OF_SCOPE#")
        or u.startswith("SHIPPING#")
        or u.startswith("SUPPORT#")
        or u.startswith("COMPANY#")
        or u.startswith("WARRANTY#")
        or u.startswith("MULTIMEDIA_DEVICES#")
        or u.startswith("HARDWARE#")
    )

    if (domain == "phone" and phone_exclusions) or (domain == "laptop" and laptop_exclusions):
        return "", "", "no", "excluded"

    return "", "", "review", "pending"


In [16]:

label_counts = (
    raw_triplets_df.groupby(["domain", "source_aspect_category"])
    .size()
    .reset_index(name="count")
    .rename(columns={"source_aspect_category": "source_label"})
)

mapping_df = label_counts.copy()

mapped_values = mapping_df.apply(
    lambda row: taxonomy_mapping_v02(row["domain"], row["source_label"]),
    axis=1,
)

mapping_df[
    ["canonical_aspect", "semantic_group", "include", "mapping_status"]
] = pd.DataFrame(mapped_values.tolist(), index=mapping_df.index)

mapping_df = mapping_df.sort_values(
    ["domain", "count"], ascending=[True, False]
).reset_index(drop=True)

mapping_path = TAXONOMY_DIR / "mabsa_taxonomy_mapping_v0_2.csv"
mapping_df.to_csv(mapping_path, index=False)

duplicate_mappings = mapping_df.duplicated(["domain", "source_label"]).sum()
included_missing_aspect = (
    mapping_df["include"].eq("yes")
    & mapping_df["canonical_aspect"].astype(str).str.strip().eq("")
).sum()
included_missing_group = (
    mapping_df["include"].eq("yes")
    & mapping_df["semantic_group"].astype(str).str.strip().eq("")
).sum()
unresolved_labels = mapping_df["include"].eq("review").sum()

mapped_triplets_df = raw_triplets_df.merge(
    mapping_df[
        [
            "domain",
            "source_label",
            "canonical_aspect",
            "semantic_group",
            "include",
            "mapping_status",
        ]
    ],
    left_on=["domain", "source_aspect_category"],
    right_on=["domain", "source_label"],
    how="left",
    validate="many_to_one",
)

unmapped_count = mapped_triplets_df["include"].isna().sum()

display(
    mapping_df.groupby(["domain", "include"])
    .size()
    .rename("source_label_count")
    .reset_index()
)

print("Unresolved labels:", unresolved_labels)
print("Unmapped annotations:", unmapped_count)


,domain,include,source_label_count
0,laptop,no,29
1,laptop,yes,79
2,phone,no,35
3,phone,yes,51


Unresolved labels: 0
Unmapped annotations: 0


In [17]:

canonical_taxonomy = (
    mapping_df[mapping_df["include"].eq("yes")]
    [["canonical_aspect", "semantic_group"]]
    .drop_duplicates()
    .sort_values(["semantic_group", "canonical_aspect"])
    .reset_index(drop=True)
)

taxonomy_dict = {
    "version": "0.2",
    "status": "operational",
    "groups": {
        group: canonical_taxonomy.loc[
            canonical_taxonomy["semantic_group"].eq(group),
            "canonical_aspect",
        ].tolist()
        for group in ["COMMON", "PHONE_SPECIFIC", "LAPTOP_SPECIFIC"]
    },
    "notes": {
        "null_aspect_term": (
            "NULL aspect terms are retained as an observed annotation property "
            "and are not automatically treated as implicit-aspect gold labels."
        )
    },
}

taxonomy_yaml_path = TAXONOMY_DIR / "aspect_taxonomy_v0_2.yaml"

with open(taxonomy_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(taxonomy_dict, f, sort_keys=False, allow_unicode=True)

display(canonical_taxonomy)


,canonical_aspect,semantic_group
0,battery,COMMON
1,charging_power,COMMON
2,connectivity,COMMON
3,cpu,COMMON
4,design,COMMON
5,display,COMMON
6,durability,COMMON
7,memory,COMMON
8,performance,COMMON
9,price,COMMON


In [18]:

aspect_sentiment_df = (
    mapped_triplets_df[mapped_triplets_df["include"].eq("yes")]
    [
        [
            "review_id",
            "domain",
            "split",
            "text",
            "aspect_term",
            "source_aspect_category",
            "canonical_aspect",
            "semantic_group",
            "sentiment",
            "is_null_aspect_term",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

sentiment_to_id = {"negative": 0, "neutral": 1, "positive": 2}
semantic_group_to_id = {"COMMON": 0, "PHONE_SPECIFIC": 1, "LAPTOP_SPECIFIC": 2}
aspect_vocabulary = sorted(aspect_sentiment_df["canonical_aspect"].unique())
aspect_to_id = {aspect: i for i, aspect in enumerate(aspect_vocabulary)}

aspect_sentiment_df["sentiment_id"] = aspect_sentiment_df["sentiment"].map(sentiment_to_id)
aspect_sentiment_df["semantic_group_id"] = aspect_sentiment_df["semantic_group"].map(semantic_group_to_id)
aspect_sentiment_df["canonical_aspect_id"] = aspect_sentiment_df["canonical_aspect"].map(aspect_to_id)

if aspect_sentiment_df[
    ["sentiment_id", "semantic_group_id", "canonical_aspect_id"]
].isna().any().any():
    raise ValueError("Label encoding failed.")

review_groups = (
    aspect_sentiment_df.groupby("review_id")
    .agg(
        canonical_aspects=("canonical_aspect", lambda x: sorted(set(x))),
        semantic_groups=("semantic_group", lambda x: sorted(set(x))),
    )
    .reset_index()
)

multilabel_reviews_df = (
    reviews_df[["review_id", "domain", "split", "text"]]
    .merge(review_groups, on="review_id", how="inner", validate="one_to_one")
)

multilabel_reviews_df["aspect_ids"] = multilabel_reviews_df["canonical_aspects"].apply(
    lambda xs: [aspect_to_id[x] for x in xs]
)
multilabel_reviews_df["num_canonical_aspects"] = multilabel_reviews_df["canonical_aspects"].apply(len)
multilabel_reviews_df["is_multi_label"] = multilabel_reviews_df["num_canonical_aspects"].gt(1)

multilabel_csv_df = multilabel_reviews_df.copy()
for col in ["canonical_aspects", "semantic_groups", "aspect_ids"]:
    multilabel_csv_df[col] = multilabel_csv_df[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
    )

print("Aspect-sentiment rows:", len(aspect_sentiment_df))
print("Multi-label reviews:", len(multilabel_reviews_df))


Aspect-sentiment rows: 3742
Multi-label reviews: 2280


In [19]:

split_sets = {
    (domain, split): set(
        reviews_df.loc[
            reviews_df["domain"].eq(domain) & reviews_df["split"].eq(split),
            "text_sha1",
        ]
    )
    for domain in ["phone", "laptop"]
    for split in ["train", "dev", "test"]
}

overlap_rows = []
for domain in ["phone", "laptop"]:
    for a, b in [("train", "dev"), ("train", "test"), ("dev", "test")]:
        overlap_rows.append({
            "domain": domain,
            "split_a": a,
            "split_b": b,
            "duplicate_text_count": len(split_sets[(domain, a)] & split_sets[(domain, b)]),
        })

split_overlap_summary = pd.DataFrame(overlap_rows)

retention = (
    reviews_df[["review_id", "domain", "split"]]
    .merge(
        multilabel_reviews_df[["review_id"]].assign(retained=True),
        on="review_id",
        how="left",
    )
)
retention["retained"] = retention["retained"].fillna(False).astype(bool)

retention_summary = (
    retention.groupby(["domain", "split"])
    .agg(
        total_reviews=("review_id", "count"),
        retained_reviews=("retained", "sum"),
    )
    .reset_index()
)
retention_summary["retention_rate"] = (
    retention_summary["retained_reviews"]
    / retention_summary["total_reviews"]
    * 100
).round(2)

display(split_overlap_summary)
display(retention_summary)


,domain,split_a,split_b,duplicate_text_count
0,phone,train,dev,0
1,phone,train,test,0
2,phone,dev,test,0
3,laptop,train,dev,2
4,laptop,train,test,1
5,laptop,dev,test,0


,domain,split,total_reviews,retained_reviews,retention_rate
0,laptop,dev,326,218,66.87
1,laptop,test,532,350,65.79
2,laptop,train,1264,840,66.46
3,phone,dev,307,116,37.79
4,phone,test,522,223,42.72
5,phone,train,1276,533,41.77


In [20]:

aspect_sentiment_path = PROCESSED_DIR / "mabsa_aspect_sentiment.csv"
multilabel_path = PROCESSED_DIR / "mabsa_multilabel_reviews.csv"
label_mapping_path = PROCESSED_DIR / "label_mappings.json"

aspect_sentiment_df.to_csv(aspect_sentiment_path, index=False)
multilabel_csv_df.to_csv(multilabel_path, index=False)

with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "aspect_to_id": aspect_to_id,
            "sentiment_to_id": sentiment_to_id,
            "semantic_group_to_id": semantic_group_to_id,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

audit_summary = dataset_summary.merge(
    retention_summary,
    on=["domain", "split"],
    how="left",
)

audit_summary.to_csv(
    RESULTS_DIR / "data_audit_summary.csv",
    index=False,
)

processed_summary = pd.DataFrame([
    {
        "dataset": "aspect_sentiment",
        "rows": len(aspect_sentiment_df),
        "phone_rows": int(aspect_sentiment_df["domain"].eq("phone").sum()),
        "laptop_rows": int(aspect_sentiment_df["domain"].eq("laptop").sum()),
    },
    {
        "dataset": "multilabel_reviews",
        "rows": len(multilabel_reviews_df),
        "phone_rows": int(multilabel_reviews_df["domain"].eq("phone").sum()),
        "laptop_rows": int(multilabel_reviews_df["domain"].eq("laptop").sum()),
    },
])

processed_summary.to_csv(
    RESULTS_DIR / "processed_data_summary.csv",
    index=False,
)

print("Essential outputs saved.")


Essential outputs saved.


In [21]:
unresolved = mapping_df[
    mapping_df["include"].eq("review")
][
    [
        "domain",
        "source_label",
        "count",
        "canonical_aspect",
        "semantic_group",
        "mapping_status",
    ]
]

display(unresolved)

,domain,source_label,count,canonical_aspect,semantic_group,mapping_status


In [22]:

required_outputs = [
    TAXONOMY_DIR / "mabsa_taxonomy_mapping_v0_2.csv",
    TAXONOMY_DIR / "aspect_taxonomy_v0_2.yaml",
    PROCESSED_DIR / "mabsa_aspect_sentiment.csv",
    PROCESSED_DIR / "mabsa_multilabel_reviews.csv",
    PROCESSED_DIR / "label_mappings.json",
    RESULTS_DIR / "data_audit_summary.csv",
    RESULTS_DIR / "processed_data_summary.csv",
]

checks = {
    "all_raw_files_exist": all(p.exists() for p in required_raw_files),
    "all_rows_parsed": reviews_df["parse_status"].eq("ok").all(),
    "no_invalid_triplets": invalid_triplet_count == 0,
    "sentiment_space_valid": set(sentiment_labels) == {"negative", "neutral", "positive"},
    "no_duplicate_taxonomy_mappings": duplicate_mappings == 0,
    "no_missing_canonical_aspects": included_missing_aspect == 0,
    "no_missing_semantic_groups": included_missing_group == 0,
    "no_unresolved_taxonomy_labels": unresolved_labels == 0,
    "no_unmapped_annotations": unmapped_count == 0,
    "all_multilabel_reviews_have_aspects": multilabel_reviews_df["num_canonical_aspects"].gt(0).all(),
    "required_outputs_saved": all(p.exists() for p in required_outputs),
}

validation_summary = pd.DataFrame(
    [{"check": k, "passed": bool(v)} for k, v in checks.items()]
)

validation_summary.to_csv(
    RESULTS_DIR / "pipeline_validation_summary.csv",
    index=False,
)

display(validation_summary)

if not validation_summary["passed"].all():
    failed = validation_summary.loc[~validation_summary["passed"], "check"].tolist()
    raise ValueError(f"C1 DATA PREPARATION PIPELINE FAILED: {failed}")

print("C1 DATA PREPARATION PIPELINE: PASSED")


,check,passed
0,all_raw_files_exist,True
1,all_rows_parsed,True
2,no_invalid_triplets,True
3,sentiment_space_valid,True
4,no_duplicate_taxonomy_mappings,True
5,no_missing_canonical_aspects,True
6,no_missing_semantic_groups,True
7,no_unresolved_taxonomy_labels,True
8,no_unmapped_annotations,True
9,all_multilabel_reviews_have_aspects,True


C1 DATA PREPARATION PIPELINE: PASSED
